In [5]:
import os
import cv2 as cv
import numpy as np
from pathlib import Path
from tqdm import tqdm
from skimage.feature import hog, local_binary_pattern


x_maskeds = np.load("../Data/Processed/x_resized_masked.npy")
y_labels = np.load("../Data/Processed/y_labels.npy")
def extract_features(img):
    features=[]
    lung_pixels = img[img>0]
    if len(lung_pixels) > 0:
        mean_val = np.mean(lung_pixels)
        std_val = np.std(lung_pixels)
        median_val = np.median(lung_pixels)
    else:
        mean_val = 0
        std_val = 0
        median_val = 0
    features.extend([mean_val, std_val, median_val])

    lbp = local_binary_pattern(image=img, P=8, R=1, method="nri_uniform")
    lbp_hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, 60), density=True)
    features.extend(lbp_hist)

    hog_feats = hog(img, orientations= 8, pixels_per_cell=(16,16), cells_per_block=(1,1), visualize=False)
    features.extend(hog_feats)
    return np.array(features)

print("Đang trích xuất đặc trưng...")
x_features=[]
for img in tqdm(x_maskeds):
    feat = extract_features(img=img)
    x_features.append(feat)

x_features = np.array(x_features, dtype=np.float32)
x_features = x_features.reshape(len(x_features), -1)

print (f"Kích thước ma trận đặc trưng là: {x_features.shape}")

os.makedirs("../Data/Features", exist_ok=True)
np.save("../Data/Features/x_features.npy", x_features)
np.save("../Data/Features/y_labels.npy", y_labels)

Đang trích xuất đặc trưng...


100%|██████████| 21165/21165 [01:06<00:00, 319.35it/s]


Kích thước ma trận đặc trưng là: (21165, 574)
